In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import joblib
from collections import deque, Counter
from ultralytics import YOLO

# ==============================================================================
# 1. CARREGAMENTO DOS MODELOS
# ==============================================================================
# Classificador de Libras (RF treinado no coletor com 126 features)
caminho_modelo_rf = "modelo_asl_rf.pkl"
dados_modelo = joblib.load(caminho_modelo_rf)
clf_libras = dados_modelo["modelo"]
colunas_esperadas = dados_modelo["colunas"]
num_features = dados_modelo["num_features"]

# Detector de objetos gerais (YOLOv8n pré-treinado no COCO)
modelo_yolo = YOLO("yolov8n.pt")

# Fila temporal para suavização do texto de Libras
historico_sinais = deque(maxlen=7)

# ==============================================================================
# 2. INICIALIZAÇÃO MEDIAPIPE E WEBCAM
# ==============================================================================
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    raise RuntimeError("Erro ao acessar a webcam.")

print("Sistema unificado iniciado. Pressione 'q' para sair.")

# ==============================================================================
# 3. LOOP INTEGRADO
# ==============================================================================
while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # --------------------------------------------------------------------------
    # ESTÁGIO 1: YOLOv8 (Detecção de Objetos)
    # --------------------------------------------------------------------------
    # verbose=False evita poluir o terminal; conf=0.45 filtra falsos positivos
    resultados_yolo = modelo_yolo(frame, conf=0.45, verbose=False)[0]
    
    caixas_objetos = []
    for box in resultados_yolo.boxes:
        cls_id = int(box.cls[0])
        nome_classe = modelo_yolo.names[cls_id]

        # Ignoramos a classe 'person' para focar apenas em objetos manipuláveis
        if nome_classe == "person":
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        caixas_objetos.append({"box": (x1, y1, x2, y2), "nome": nome_classe, "conf": conf})

        # Desenha a caixa do objeto em azul
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 180, 0), 2)
        cv2.putText(frame, f"{nome_classe} {conf*100:.0f}%", (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 180, 0), 2)

    # --------------------------------------------------------------------------
    # ESTÁGIO 2: MediaPipe + Random Forest (Libras)
    # --------------------------------------------------------------------------
    resultado_hands = hands.process(frame_rgb)
    sinal_estavel = "Aguardando sinal..."
    cor_libras = (0, 0, 255)
    centros_maos = []

    if resultado_hands.multi_hand_landmarks:
        maos_ordenadas = sorted(resultado_hands.multi_hand_landmarks, key=lambda m: m.landmark[0].x)
        coordenadas = []

        for hand_landmarks in maos_ordenadas[:2]:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

            pulso = hand_landmarks.landmark[0]
            pulso_x, pulso_y, pulso_z = pulso.x, pulso.y, pulso.z

            # Guarda o ponto central estimado da mão em pixels (usando o nó 9 da palma)
            centro_px = (int(hand_landmarks.landmark[9].x * w), int(hand_landmarks.landmark[9].y * h))
            centros_maos.append(centro_px)

            for lm in hand_landmarks.landmark:
                coordenadas.extend([lm.x - pulso_x, lm.y - pulso_y, lm.z - pulso_z])

        if len(maos_ordenadas) == 1 and num_features == 126:
            coordenadas.extend([0.0] * 63)

        coordenadas = coordenadas[:num_features]
        X_input = pd.DataFrame([coordenadas], columns=colunas_esperadas)

        predicao = clf_libras.predict(X_input)[0]
        probabilidades = clf_libras.predict_proba(X_input)[0]
        conf_libras = np.max(probabilidades)

        if conf_libras >= 0.45:
            historico_sinais.append(predicao)
            sinal_estavel = Counter(historico_sinais).most_common(1)[0][0]
            cor_libras = (0, 220, 0)
        else:
            sinal_estavel = "Incerto"
            cor_libras = (0, 165, 255)
    else:
        historico_sinais.clear()

    # --------------------------------------------------------------------------
    # ESTÁGIO 3: Associação Espacial (Objeto na Mão)
    # --------------------------------------------------------------------------
    objeto_na_mao = None
    for centro in centros_maos:
        cx, cy = centro
        for obj in caixas_objetos:
            ox1, oy1, ox2, oy2 = obj["box"]
            # Margem de tolerância ao redor da caixa do objeto
            margem = 20
            if (ox1 - margem <= cx <= ox2 + margem) and (oy1 - margem <= cy <= oy2 + margem):
                objeto_na_mao = obj["nome"]
                break
        if objeto_na_mao:
            break

    # --------------------------------------------------------------------------
    # HUD / Painel Informativo Superior
    # --------------------------------------------------------------------------
    cv2.rectangle(frame, (10, 10), (630, 80), (30, 30, 30), -1)

    # Status de Libras
    cv2.putText(frame, f"Libras: {sinal_estavel}", (20, 42),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, cor_libras, 2)

    # Status do objeto em mãos
    txt_objeto = f"Segurando: {objeto_na_mao}" if objeto_na_mao else "Mao livre"
    cor_obj_txt = (0, 255, 255) if objeto_na_mao else (200, 200, 200)
    cv2.putText(frame, txt_objeto, (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, cor_obj_txt, 2)

    cv2.imshow("Sistema Integrado: Libras + Objetos", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

FileNotFoundError: [Errno 2] No such file or directory: 'modelo_asl_rf.pkl'